# 3. GPT-2 training and test inference

In [ ]:
%pip install accelerate -U
%pip install transformers[torch]

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    LineByLineTextDataset,
    TrainingArguments,
    Trainer,
)

import pandas as pd

In [ ]:
KAGGLE_TRAIN = False
TRAIN_SIZE = 0.25

In [ ]:
if KAGGLE_TRAIN:
    !wget https://media.githubusercontent.com/media/danielpancake/text-detox/main/data/interim/processed.tsv

    train_data_file = "/kaggle/working/processed.tsv"
    model_output_dir = "/kaggle/working/models/gpt2-based"

    TRAIN_COMBINED_PATH = "/kaggle/working/train.txt"
else:
    train_data_file = "../data/interim/processed.tsv"
    model_output_dir = "../models/gpt2-based"

    TRAIN_COMBINED_PATH = "../data/temporary/train.txt"

In [ ]:
df = pd.read_csv(train_data_file, sep="\t", header=None, names=["tox", "detox"])
df.head()

In [ ]:
# The format is: [TOX]text[/TOX]»»[DETOX]text[/DETOX]
# [TOX]   - source text
# [DETOX] - target text
# »»      - separator

# So, we will add 5 custom tokens to the vocabulary:
# [TOX], [/TOX], [DETOX], [/DETOX], »»
tokens_dict = {
    "tox_begin": "[TOX]",
    "tox_end": "[/TOX]",
    "detox_begin": "[DETOX]",
    "detox_end": "[/DETOX]",
    "separator": "»»",
    "split": "[SPLIT]",
}

In [ ]:
# Make a combined column of the toxic and detoxified sentences with the special tokens
df["combined"] = (
    tokens_dict["tox_begin"]
    + df["tox"]
    + tokens_dict["tox_end"]
    + tokens_dict["separator"]
    + tokens_dict["detox_begin"]
    + df["detox"]
    + tokens_dict["detox_end"]
)

# Make a sample of the data for training with size TRAIN_SIZE * len(df)
df_train = df.sample(frac=TRAIN_SIZE, random_state=42)
df_train["combined"].to_csv(TRAIN_COMBINED_PATH, index=False, header=False)
df_train.head()

In [ ]:
# Train size
f"Train will be of {len(df_train)} samples out of {len(df)}"

In [ ]:
batch_size = 8
epochs = 1

# For the block size, we use a length of the longest sentence in our dataset
block_size = df_train["combined"].map(len).max() + 1

warmup_steps = 500
save_steps = 10_000
logging_steps = 500

In [ ]:
model = AutoModelForCausalLM.from_pretrained("gpt2", cache_dir="cache")
tokenizer = AutoTokenizer.from_pretrained("gpt2", cache_dir="cache")
datacollator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

In [ ]:
num_added_toks = tokenizer.add_tokens(list(tokens_dict.values()))
print(f"Added {num_added_toks} tokens")
model.resize_token_embeddings(len(tokenizer), pad_to_multiple_of=8)

In [ ]:
train_dataset = LineByLineTextDataset(
    tokenizer=tokenizer,
    file_path=TRAIN_COMBINED_PATH,
    block_size=block_size
)

In [ ]:
training_args = TrainingArguments(
    output_dir=model_output_dir,
    overwrite_output_dir=True,
    num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    warmup_steps=warmup_steps,
    save_steps=save_steps,
    logging_steps=logging_steps,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=datacollator,
    train_dataset=train_dataset,
)

trainer.train()

# Save model and tokenizer
trainer.save_model(model_output_dir)
tokenizer.save_pretrained(model_output_dir)

In [ ]:
# Inference pipeline
from transformers import pipeline

generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device="cuda")

In [ ]:
def detoxify(text: str) -> str:
    prompt = (
        tokens_dict["tox_begin"]
        + text
        + tokens_dict["tox_end"]
        + tokens_dict["separator"]
        + tokens_dict["detox_begin"]
    )

    output = generator(prompt, max_length=len(prompt) * 2.5)[0]["generated_text"]

    suggestions = output.split(tokens_dict["separator"] + tokens_dict["detox_begin"])[1]
    
    # Replace all the special tokens with [SPLIT]
    for token in tokens_dict.values():
        suggestions = suggestions.replace(token, tokens_dict["split"])

    # Split by [SPLIT]
    suggestions = suggestions.split(tokens_dict["split"])

    # Trim if any of \", \n, or whitespace is present
    suggestions = [s.strip('"\n ') for s in suggestions]

    # Remove empty strings
    suggestions = list(filter(None, suggestions))

    return suggestions

In [ ]:
prompt = "I hate your stupid fucking face!!"

for suggestion in detoxify(prompt):
    print(suggestion)